# Fairness : Mi-Projet

*Travail réalisé par Cyprien BRION - L3 Informatique*

## Introduction

L'essor des méthodes d'apprentissage automatique dans le domaine médical soulève des enjeux majeurs en matière d'équité et de biais algorithmiques. En effet, les modèles entraînés sur des données de santé peuvent reproduire, voire amplifier, des biais présents dans les jeux de données d'origine, liés par exemple à l'âge, au genre ou à d'autres caractéristiques démographiques des patients. Ces biais peuvent donc conduire à des performances inégales selon les populations et poser de nombreux problèmes éthiques.

Dans ce mi-projet, nous nous intéresserons au jeu de données NIH Chest X-ray 14, qui regroupe des examens de radiographies thoraciques accompagnés de métadonnées décrivant les patients. Dans un premier temps, nous nous concentrerons exclusivement sur ces métadonnées, sans exploiter les images, afin d'analyser la structure du jeu de données et d'identifier d'éventuels biais statistiques.

Dans ce notebook, nous réaliserons une analyse descriptive approfondie des métadonnées afin de mettre en évidence des biais potentiels entre différents groupes d'individus. Ensuite, nous réaliserons une méthode de mitigation des biais par pré-processing visant à atténuer ces biais, tout en conservant autant que possible l'information utile contenue dans les données.

## I. Préparation de la donnée

### a. Chargement de données et aperçu des données

Commençons par charger notre jeu de données :

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Pour le chargement du jeu de données
df = pd.read_csv("Brion_Cyprien.csv")
df.head()


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171
1,00000004_000.png,Mass|Nodule,0,4,82,M,AP,2500,2048,0.168,0.168
2,00000005_000.png,No Finding,0,5,69,F,PA,2048,2500,0.168,0.168
3,00000005_001.png,No Finding,1,5,69,F,AP,2500,2048,0.168,0.168
4,00000005_002.png,No Finding,2,5,69,F,AP,2500,2048,0.168,0.168


Ensuite, observons plus en détail les caractéristiques de ce jeu de données :

In [3]:
df.shape

(53951, 11)

Analysons désormais le type des variables contenues dans ce dataset :

In [4]:
df.dtypes

Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
dtype: object

### b. Vérifications et nettoyage du jeu de données

Vérifions désormais la présence de valeurs manquantes dans le jeu de données.

In [5]:
df.isna().sum()

Image Index                    0
Finding Labels                 0
Follow-up #                    0
Patient ID                     0
Patient Age                    0
Patient Gender                 0
View Position                  0
OriginalImage[Width            0
Height]                        0
OriginalImagePixelSpacing[x    0
y]                             0
dtype: int64

On observe qu'aucune des variables du jeu de données ne contient de valeurs manquantes.

Regardons désormais si notre jeu de données contient des valeurs aberrantes. Bien sûr, cela ne concerne que les variables numériques.

In [6]:
df.describe()

,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
count,53951.000000,53951.000000,53951.000000,53951.000000,53951.000000,53951.000000,53951.000000
mean,8.432263,14409.935367,46.937054,2645.894163,2488.074790,0.155670,0.155670
std,15.228100,8371.393985,16.712377,341.299413,402.836045,0.016239,0.016239
min,0.000000,2.000000,1.000000,1143.000000,1001.000000,0.115000,0.115000
25%,0.000000,7526.000000,35.000000,2500.000000,2048.000000,0.143000,0.143000
50%,3.000000,14006.000000,49.000000,2520.000000,2544.000000,0.143000,0.143000
75%,9.000000,20725.000000,59.000000,2992.000000,2991.000000,0.168000,0.168000
max,172.000000,30805.000000,413.000000,3550.000000,4715.000000,0.198800,0.198800


On remarque ici qu'il y a des valeurs aberrantes pour la colonne `Patient Age`. En effet, le maximum pour cette colonne est de 413, ce qui signifierait qu'un patient à 413 ans, ce qui n'est clairement pas réaliste.

Pour corriger ce problème, nous allons filtrer les valeurs de la colonne `Patient Age` selon un intervalle réaliste : 

In [10]:
df = df[(df["Patient Age"] >= 0) & (df["Patient Age"] <= 120)]
df["Patient Age"].describe()

count    53942.000000
mean        46.900467
std         16.385627
min          1.000000
25%         35.000000
50%         49.000000
75%         59.000000
max         94.000000
Name: Patient Age, dtype: float64

Après nettoyage, les valeurs sont désormais bien plus réalistes.

Effectuons une analyse pour observer le nombre de valeurs distinctes par variable afin de confirmer qu'il n'y aurait pas de valeurs manquantes encodées de manière implicite.

In [11]:
df.nunique()

Image Index                    53942
Finding Labels                   627
Follow-up #                      173
Patient ID                     15000
Patient Age                       94
Patient Gender                     2
View Position                      2
OriginalImage[Width              756
Height]                          912
OriginalImagePixelSpacing[x       21
y]                                21
dtype: int64

On remarque donc que le nombre de valeurs distinctes est cohérent avec l'énoncé. Aucune variable ne présente donc de valeurs manquantes qui auraient pu être encodées implicitement.
Néanmoins, à partir de ces résultats on peut noter que :
* Les variables `Patient Gender` et `View Position` ne possèdent que deux valeurs distinctes chacune, ce qui en fait des variables clés pour l'analyse des biais.

Pour finir, vérifions si des lignes ont été dupliquées :

In [12]:
dup_rows = df.duplicated().sum()
print(f"Doublons exacts (toutes colonnes) : {dup_rows}")

Doublons exacts (toutes colonnes) : 0


Il n'y a donc aucun doublon dans notre dataset.

Observons la distribution du nombre d'examens par patient afin de repérer les patients multi-examens :

In [14]:
exam_stats = df.groupby("Patient ID").size().describe()
print("Statistiques du nombre d'examens par patient :")
print(exam_stats)

Statistiques du nombre d'examens par patient :
count    15000.000000
mean         3.596133
std          7.163112
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        173.000000
dtype: float64


Ceci va nous permettre de répondre à la question suivante : les patients qui ont beaucoup d'examens sont-ils plus souvent associés à certaines pathologies ?

In [18]:
exam_counts = df.groupby("Patient ID").size()
exam_counts.describe()

count    15000.000000
mean         3.596133
std          7.163112
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        173.000000
dtype: float64

On découpe les patients selon leur nombre d'examens et on calcule pour chaque groupe, la proportion de patiens ayant au moins une pathologie avec `has_disease = True`

In [19]:
df = df.merge(
    exam_counts.rename("nb_exams"),
    left_on="Patient ID",
    right_index=True
)


In [20]:
df["has_disease"] = df["Finding Labels"] != "No Finding"

In [22]:
df.groupby(
    pd.cut(df["nb_exams"], bins=[0,1,3,10,200])
)["has_disease"].mean()


/tmp/ipykernel_12227/1971727585.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


nb_exams
(0, 1]       0.272535
(1, 3]       0.353521
(3, 10]      0.434452
(10, 200]    0.579562
Name: has_disease, dtype: float64

On observe une relation strictement croissante ce qui est plutôt logique. Les patients qui reviennent souvent sont beaucoup plus souvent malades. Le nombre d'examens est donc corrélé à l'état de santé.

**Remarque :** cette analyse n'est pas un biais du modèle mais cela pourra être problématique pour l'apprentissage du modèle. Nous n'irons pas plus loin pour cette analyse puisque l'apprentissage du modèle n'est pas demandé.

## II. Analyse descriptive et observation des biais